# Split Vietnam License Plate dataset into train/val/test (80/10/10)

Dataset: `Vietnam License plate.v4i.yolov8` (currently everything is in `train/`).
This notebook moves a random 10% of pairs into `valid/` and another 10% into `test/`,
then writes a `data.yaml` ready for YOLO fine-tuning.

In [ ]:
import random
import shutil
from pathlib import Path

ROOT = Path("Vietnam License plate.v4i.yolov8").resolve()
TRAIN_IMG = ROOT / "train" / "images"
TRAIN_LBL = ROOT / "train" / "labels"

SEED = 42
VAL_RATIO = 0.10
TEST_RATIO = 0.10

print("Root:", ROOT)
print("Images in train:", len(list(TRAIN_IMG.glob("*.jpg"))))

In [ ]:
# Create destination folders
for split in ("valid", "test"):
    (ROOT / split / "images").mkdir(parents=True, exist_ok=True)
    (ROOT / split / "labels").mkdir(parents=True, exist_ok=True)

In [ ]:
# Pick a random subset of image/label pairs for valid and test
images = sorted(p for p in TRAIN_IMG.iterdir() if p.suffix.lower() in (".jpg", ".jpeg", ".png"))
random.seed(SEED)
random.shuffle(images)

n_total = len(images)
n_val = round(n_total * VAL_RATIO)
n_test = round(n_total * TEST_RATIO)

val_images = images[:n_val]
test_images = images[n_val:n_val + n_test]

print(f"total={n_total}  val={len(val_images)}  test={len(test_images)}  train(remaining)={n_total - n_val - n_test}")

In [ ]:
def move_pairs(image_paths, split_name):
    moved = 0
    for img_path in image_paths:
        lbl_path = TRAIN_LBL / (img_path.stem + ".txt")
        if not lbl_path.exists():
            print("Missing label for", img_path.name)
            continue
        shutil.move(str(img_path), ROOT / split_name / "images" / img_path.name)
        shutil.move(str(lbl_path), ROOT / split_name / "labels" / lbl_path.name)
        moved += 1
    print(f"Moved {moved} pairs to {split_name}")

move_pairs(val_images, "valid")
move_pairs(test_images, "test")

In [ ]:
# Sanity check final counts
for split in ("train", "valid", "test"):
    n_img = len(list((ROOT / split / "images").glob("*")))
    n_lbl = len(list((ROOT / split / "labels").glob("*")))
    print(f"{split}: images={n_img} labels={n_lbl}")

In [ ]:
# Write data.yaml ready for YOLO fine-tuning
data_yaml = f"""path: {ROOT}
train: train/images
val: valid/images
test: test/images

nc: 1
names:
  0: license_plate
"""

(ROOT / "data.yaml").write_text(data_yaml)
print(data_yaml)